# Ch 23. KV Cache — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Compare the time to generate 100 tokens with the KV cache on/off at $n=128,512,2048$.

### Solution

Without a cache, every decode step recomputes K,V for the full prefix, with work proportional to $\sum_{t=n}^{n+99}t$. With a cache, only the new token is projected, proportional to 100. Since latency is hardware-dependent, the code reports exact work proxies and scaling.


In [ ]:
import time
import numpy as np

# Reduced autoregressive projection benchmark: recompute the prefix or append one cached row.
rng=np.random.default_rng(2301); width=64; generated=100; W=rng.normal(size=(width,width)); results={}
for initial in (128,512,2048):
    tokens=rng.normal(size=(initial+generated,width))
    start=time.perf_counter();
    for step in range(generated): tokens[:initial+step+1]@W
    no_cache=time.perf_counter()-start
    start=time.perf_counter(); cache=tokens[:initial]@W
    for step in range(generated): cache=np.vstack((cache,tokens[initial+step:initial+step+1]@W))
    cached=time.perf_counter()-start
    results[initial]={"no_cache_ms":1000*no_cache,"cache_ms":1000*cached,"speedup":no_cache/cached}
assert all(r["speedup"]>1 for r in results.values())
print({n:{k:round(v,3) for k,v in r.items()} for n,r in results.items()})


## Problem 2

**Problem**: Calculate KV-cache memory for a 32K context in LLaMA-7B.

### Solution

LLaMA-7B has 32 layers, 32 KV heads, and head dimension 128. In FP16, memory is $2\times32\times32\times128\times32768\times2$ bytes $=16$ GiB. The first 2 is K and V; the last is bytes per element.


In [ ]:
layers,kv_heads,head_dim,context,batch,bytes_per_value=32,32,128,32768,1,2
bytes_used=2*layers*kv_heads*head_dim*context*batch*bytes_per_value
gib=bytes_used/1024**3
assert bytes_used==17_179_869_184 and gib==16.0
print({"layers":layers,"kv_heads":kv_heads,"head_dim":head_dim,"context":context,"batch":batch,
       "dtype":"fp16","KV_cache_bytes":bytes_used,"KV_cache_GiB":gib})


## Problem 3

**Problem**: Compare how much KV-cache memory GQA saves for LLaMA-7B (32 heads) versus LLaMA-70B (8 KV heads).

### Solution

At equal layers, head dimension, and length, cache size is linear in KV-head count. Reducing 32 to 8 yields one quarter, a 75% saving. Actual 7B and 70B models have different layer counts, which must be included in total-memory comparisons.
### Explanation

The named-model comparison must include layer count. The 7B model has $32\times32=1024$ layer-heads, while the 70B model has $80\times8=640$, so at equal context, head dimension, and dtype the 70B total KV cache is 62.5% of the 7B cache: a 37.5% saving. Separately, holding an 80-layer architecture fixed and changing only KV heads from 32 to 8 saves 75%.


In [ ]:
# Named-model total includes each architecture's layer count; fixed-architecture isolates head sharing.
batch,context,head_dim,bytes_per_value=1,32768,128,2
def kv_bytes(layers,heads): return 2*batch*layers*heads*context*head_dim*bytes_per_value
llama_7b=kv_bytes(32,32); llama_70b=kv_bytes(80,8)
named_saving=1-llama_70b/llama_7b
fixed_mha=kv_bytes(80,32); fixed_gqa=kv_bytes(80,8); fixed_saving=1-fixed_gqa/fixed_mha
assert named_saving==0.375 and fixed_saving==0.75
print({"named_models":{"LLaMA-7B_GiB":llama_7b/1024**3,"LLaMA-70B_GiB":llama_70b/1024**3,
       "70B_vs_7B_saving":named_saving},"fixed_80_layer_architecture":{"32_to_8_KV_heads_saving":fixed_saving}})


## Problem 4

**Problem**: Demonstrate the throughput improvement of continuous batching with a simulation.

### Solution

Static batching leaves slots idle until the longest request finishes. Continuous batching inserts a new request immediately upon completion, increasing active-slot time. The discrete-event simulation deterministically compares makespan and throughput for identical request lengths.


In [ ]:
import heapq

lengths=[9,2,7,3,8,4,6,5,2,9,3,7]; slots=3
static_time=sum(max(lengths[i:i+slots]) for i in range(0,len(lengths),slots))
workers=[0]*slots
for length in lengths:
    earliest=heapq.heappop(workers); heapq.heappush(workers,earliest+length)
continuous_time=max(workers)
tokens=sum(lengths); static_throughput=tokens/static_time; continuous_throughput=tokens/continuous_time
assert continuous_throughput>static_throughput
print({"requests":lengths,"static_ticks":static_time,"continuous_ticks":continuous_time,
       "static_tokens_per_tick":round(static_throughput,3),"continuous_tokens_per_tick":round(continuous_throughput,3),
       "throughput_gain":round(continuous_throughput/static_throughput,3)})


## Problem 5

**Problem**: Explain how PagedAttention resolves memory fragmentation.

### Solution

PagedAttention splits a logically contiguous KV cache into fixed-size physical blocks mapped by a page table. It avoids preallocating contiguous maximum-length space, reducing external fragmentation; internal waste is less than one final block per sequence. Block sharing also enables prefix reuse.


In [ ]:
import math

requests=[17,33,65,7]; block=16
contiguous_reserved=[64,64,128,32]
paged_reserved=[math.ceil(n/block)*block for n in requests]
contiguous_waste=sum(contiguous_reserved)-sum(requests); paged_waste=sum(paged_reserved)-sum(requests)
page_table={request:list(range(sum(paged_reserved[:i])//block,sum(paged_reserved[:i+1])//block)) for i,request in enumerate(requests)}
assert paged_waste < contiguous_waste and all(waste < block for waste in [r-n for r,n in zip(paged_reserved,requests)])
print({"request_tokens":requests,"contiguous_waste":contiguous_waste,"paged_waste":paged_waste,"page_table":page_table})
